In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving voltage_stability_final.csv to voltage_stability_final (1).csv


In [ ]:
df = pd.read_csv('voltage_stability_final.csv')
df.head()

,Bus ID,Voltage Magnitude,Voltage Angle,Load Factor,Stability Label,voltage_deviation,stress_index,is_outage
0,0,1.030957,40.330800,0.6,STABLE,0.030957,0.018574,1
1,1,1.014978,58.253833,0.6,STABLE,0.014978,0.008987,1
2,2,0.971241,52.192754,0.6,STABLE,0.028759,0.017256,1
3,3,0.922611,40.375760,0.6,STABLE,0.077389,0.046433,1
4,4,0.915772,32.559313,0.6,STABLE,0.084228,0.050537,1


In [ ]:
df['label_binary'] = df['Stability Label'].apply(
    lambda x: 0 if x == 'STABLE' else 1
)

print(df['label_binary'].value_counts())

label_binary
0    16286
1     1863
Name: count, dtype: int64


In [ ]:
df = df.sort_values(['Bus ID', 'Load Factor']).reset_index(drop=True)
print(df.shape)
print(df.head(10))

(18149, 9)
   Bus ID  Voltage Magnitude  Voltage Angle  Load Factor Stability Label  \
0       0           1.030957      40.330800          0.6          STABLE   
1       0           1.018354      55.305745          0.6          STABLE   
2       0           1.034251      62.900331          0.6         WARNING   
3       0           1.031725      46.566484          0.6          STABLE   
4       0           1.027043      66.697384          0.6        UNSTABLE   
5       0           1.026938      48.485323          0.6          STABLE   
6       0           1.025921      60.025264          0.6         WARNING   
7       0           1.028535      49.752912          0.6          STABLE   
8       0           1.029992      55.499190          0.6          STABLE   
9       0           1.030211      49.554470          0.6          STABLE   

   voltage_deviation  stress_index  is_outage  label_binary  
0           0.030957      0.018574          1             0  
1           0.018354      0.

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['Stability Label'])

print(le.classes_)
print(df['label_encoded'].value_counts())

['STABLE' 'UNSTABLE' 'WARNING']
label_encoded
0    16286
2     1439
1      424
Name: count, dtype: int64


In [ ]:
print(df.columns.tolist())

['Bus ID', 'Voltage Magnitude', 'Voltage Angle', 'Load Factor', 'Stability Label', 'voltage_deviation', 'stress_index', 'is_outage', 'label_binary', 'label_encoded']


Sequences must be from the same bus — same location, same sensor, continuous trajectory.

In [ ]:
def create_sequences_binary(df, window_size=10):
    X, y = [], []

    feature_cols = ['Voltage Magnitude', 'Voltage Angle',
                    'Load Factor', 'voltage_deviation',
                    'stress_index', 'is_outage']

    for bus_id in df['Bus ID'].unique():
        bus_data = df[df['Bus ID'] == bus_id].sort_values('Load Factor')

        features = bus_data[feature_cols].values
        labels = bus_data['label_binary'].values

        for i in range(len(features) - window_size):
            X.append(features[i:i+window_size])
            y.append(labels[i+window_size])

    return np.array(X), np.array(y)

X_seq, y_seq = create_sequences_binary(df, window_size=10)
print("X shape:", X_seq.shape)
print("y shape:", y_seq.shape)
print("Label distribution:", np.unique(y_seq, return_counts=True))

X shape: (17759, 10, 6)
y shape: (17759,)
Label distribution: (array([0, 1]), array([15951,  1808]))


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Train labels:", np.unique(y_train, return_counts=True))
print("Test labels:", np.unique(y_test, return_counts=True))

X_train shape: (14207, 10, 6)
X_test shape: (3552, 10, 6)
Train labels: (array([0, 1]), array([12761,  1446]))
Test labels: (array([0, 1]), array([3190,  362]))


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [ ]:
# y_train_cat = to_categorical(y_train, num_classes=3)
# y_test_cat = to_categorical(y_test, num_classes=3)
# earlier we used this but it didin't work so we combines unstable and warning into one class

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))
print("Class weights:", class_weight_dict)

Class weights: {np.int64(0): np.float64(0.5566570018023665), np.int64(1): np.float64(4.912517289073306)}


In [ ]:
model_binary = Sequential([
    LSTM(64, input_shape=(10, 6), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model_binary.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model_binary.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_5 (LSTM)                   │ (None, 64)             │        18,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,289 (79.25 KB)

 Trainable params: 20,289 (79.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model_binary.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    class_weight=class_weight_dict,
    verbose=1
)

Epoch 1/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8391 - loss: 0.3776 - val_accuracy: 0.8241 - val_loss: 0.3924
Epoch 2/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8379 - loss: 0.3194 - val_accuracy: 0.8578 - val_loss: 0.3296
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8401 - loss: 0.3152 - val_accuracy: 0.7994 - val_loss: 0.3800
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8403 - loss: 0.3149 - val_accuracy: 0.7706 - val_loss: 0.4137
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8403 - loss: 0.3076 - val_accuracy: 0.8473 - val_loss: 0.2953
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8413 - loss: 0.3086 - val_accuracy: 0.8529 - val_loss: 0.3227
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8485 - loss: 0.3067 - val_accuracy: 0.8473 - val_loss: 0.3383
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8532 - loss: 0.3019 - val_accuracy: 0

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix

y_pred_prob = model_binary.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred , target_names=['STABLE', 'DANGER']))
print(confusion_matrix(y_test, y_pred))

111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
              precision    recall  f1-score   support

      STABLE       0.98      0.88      0.93      3190
      DANGER       0.45      0.84      0.58       362

    accuracy                           0.88      3552
   macro avg       0.71      0.86      0.76      3552
weighted avg       0.93      0.88      0.89      3552

[[2816  374]
 [  58  304]]
